# In-Context Learning: Thinking vs Non-Thinking (qwen3:0.6b y qwen3:4b)

En este notebook evaluo el rendimiento de ICL (In-Context Learning) para atribucion de autoria,
comparando dos modelos (0.6b y 4b) en dos modos cada uno: standard (sin thinking) vs thinking (con Chain-of-Thought + contexto extendido).

## 0. Setup e Imports

In [ ]:
# Importo todas las librerias que necesito para el experimento
from pathlib import Path
import json
import datetime
import requests  # Para comunicarme con Ollama via API
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split  # Para crear subset estratificado
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report
from tqdm.auto import tqdm  # Barra de progreso visual
import warnings
warnings.filterwarnings('ignore')

# Configuro el estilo de los plots para que se vean bien
plt.style.use('seaborn-v0_8')
pd.set_option('display.max_colwidth', 120)  # Muestro hasta 120 caracteres en columnas
pd.set_option('display.precision', 3)  # 3 decimales de precision

In [ ]:
# Defino las rutas del proyecto
# Necesito encontrar la raiz del proyecto para acceder a los datos procesados

def find_root() -> Path:
    """Busco la raiz del proyecto localizando el directorio data/raw."""
    current = Path.cwd()
    # Subo por el arbol de directorios hasta encontrar data/raw
    for cand in [current, *current.parents]:
        if (cand / "data" / "raw").exists():
            return cand
    raise FileNotFoundError("No encuentro data/raw desde el directorio actual.")

# Establezco todas las rutas que voy a necesitar
PROJECT_ROOT = find_root()
DATA_RAW = PROJECT_ROOT / "data" / "raw"
DATA_PROCESSED = PROJECT_ROOT / "data" / "processed"
BOUNDARIES_DIR = DATA_PROCESSED / "boundaries"  # CSVs con pares de oraciones etiquetadas
REPORTS_DIR = PROJECT_ROOT / "reports"  # Donde guardo metricas y graficos
PREDICTIONS_DIR = DATA_PROCESSED / "predictions"  # Donde guardo las predicciones
PREDICTIONS_DIR.mkdir(parents=True, exist_ok=True)  # Creo el dir si no existe

print(f"PROJECT_ROOT: {PROJECT_ROOT}")
print(f"BOUNDARIES_DIR: {BOUNDARIES_DIR}")
print(f"REPORTS_DIR: {REPORTS_DIR}")

In [ ]:
# Configuracion del experimento
# Aqui defino todos los parametros importantes

MODELS = ["qwen3:0.6b", "qwen3:4b"]  # Los dos modelos que voy a comparar (pequeno y mediano)
N_EVAL = 500  # Cuantos ejemplos evaluo (subset estratificado de validation)
N_SHOTS = 15  # Cuantos ejemplos de cada clase incluyo en el prompt (few-shot learning)
SEED = 42  # Semilla para reproducibilidad
OLLAMA_URL = "http://localhost:11434/api/generate"  # Endpoint de Ollama local
REQUEST_TIMEOUT = 60  # Timeout en segundos para cada request
MAX_RETRIES = 3  # Cuantos reintentos si falla una peticion

# Fijo la semilla para que los resultados sean reproducibles
np.random.seed(SEED)

print(f"Configuracion del experimento:")
print(f"  Modelos: {MODELS}")
print(f"  N_EVAL: {N_EVAL}")
print(f"  N_SHOTS: {N_SHOTS} (por clase)")

### Funciones de metricas de segmentacion

Reutilizo las funciones Pk y WindowDiff del notebook 07 para evaluar la calidad de la segmentacion.

In [ ]:
# Funciones para calcular metricas de segmentacion
# Estas metricas evaluan que tan bien el modelo detecta cambios de autor

def boundaries_to_segments(y_boundaries: np.ndarray) -> np.ndarray:
    """Convierto un vector de fronteras binarias (0/1) a IDs de segmento.
    
    Por ejemplo: [0, 1, 0, 0, 1] -> [0, 0, 1, 1, 1, 2]
    Cada 1 indica un cambio de autor, asi que incremento el ID del segmento.
    """
    y_boundaries = np.asarray(y_boundaries, dtype=int)
    n_sentences = len(y_boundaries) + 1  # Tengo n+1 oraciones para n boundaries
    segments = np.zeros(n_sentences, dtype=int)
    current = 0  # Empiezo en el segmento 0
    segments[0] = current
    
    # Recorro las boundaries y cambio de segmento cuando encuentro un 1
    for i, val in enumerate(y_boundaries):
        if val:  # Si hay cambio de autor (1)
            current += 1  # Incremento el ID del segmento
        segments[i + 1] = current
    return segments


def pk(reference: np.ndarray, hypothesis: np.ndarray, k: int) -> float:
    """Calculo la metrica Pk de segmentacion.
    
    Pk mide la probabilidad de que dos oraciones separadas por k posiciones
    esten incorrectamente clasificadas como mismo/diferente segmento.
    Menor es mejor (0 = perfecto).
    """
    reference = np.asarray(reference)
    hypothesis = np.asarray(hypothesis)
    
    if len(reference) != len(hypothesis):
        raise ValueError('reference y hypothesis deben tener la misma longitud')
    if k <= 0:
        raise ValueError('k debe ser positivo')
    if len(reference) <= k:
        return 0.0
    
    disagreements = 0
    total = 0
    
    # Comparo cada par de oraciones separadas por k posiciones
    for i in range(len(reference) - k):
        same_ref = reference[i] == reference[i + k]  # Mismo segmento en referencia?
        same_hyp = hypothesis[i] == hypothesis[i + k]  # Mismo segmento en prediccion?
        # Si no coinciden, cuento un error
        disagreements += int(same_ref != same_hyp)
        total += 1
    
    return disagreements / total if total else 0.0


def windowdiff(reference: np.ndarray, hypothesis: np.ndarray, k: int) -> float:
    """Calculo la metrica WindowDiff de segmentacion.
    
    WindowDiff cuenta cuantas ventanas de tamano k tienen un numero diferente
    de boundaries entre la referencia y la hipotesis.
    Menor es mejor (0 = perfecto).
    """
    reference = np.asarray(reference, dtype=int)
    hypothesis = np.asarray(hypothesis, dtype=int)
    
    if reference.shape != hypothesis.shape:
        raise ValueError('reference y hypothesis deben tener la misma longitud')
    if k <= 0:
        raise ValueError('k debe ser positivo')
    
    n_sentences = len(reference) + 1
    if n_sentences <= k:
        return 0.0
    
    total = n_sentences - k + 1  # Numero total de ventanas
    errors = 0
    
    # Cuento boundaries en cada ventana y comparo
    for start in range(total):
        end = start + k - 1
        ref_count = reference[start:end].sum()  # Boundaries en ref
        hyp_count = hypothesis[start:end].sum()  # Boundaries en hyp
        # Si el numero de boundaries difiere, es un error
        errors += int(ref_count != hyp_count)
    
    return errors / total if total else 0.0


def evaluate_predictions(boundaries_df: pd.DataFrame, y_pred: np.ndarray) -> dict:
    """Evaluo las predicciones con metricas de clasificacion y segmentacion.
    
    Calculo:
    - Metricas de clasificacion: accuracy, precision, recall, F1
    - Metricas de segmentacion: Pk, WindowDiff
    """
    # Ordeno por nivel, documento y boundary para que todo este consistente
    df = boundaries_df.sort_values(["level", "doc_id", "boundary_id"]).reset_index(drop=True)
    y_true = df["y"].to_numpy()
    y_pred = np.asarray(y_pred, dtype=int)
    
    if len(y_pred) != len(y_true):
        raise ValueError("y_pred debe tener la misma longitud que y_true")
    
    # Calculo metricas de clasificacion (tratan cada boundary independientemente)
    accuracy = accuracy_score(y_true, y_pred)
    precision_macro = precision_score(y_true, y_pred, average="macro", zero_division=0)
    recall_macro = recall_score(y_true, y_pred, average="macro", zero_division=0)
    f1_macro = f1_score(y_true, y_pred, average="macro", zero_division=0)
    
    # Calculo k para las metricas de segmentacion
    # k debe ser aproximadamente la mitad de la longitud media de los segmentos
    segment_lengths = []
    for (_, _), group in df.groupby(["level", "doc_id"]):
        y_doc = group["y"].to_numpy()
        seg = boundaries_to_segments(y_doc)
        _, counts = np.unique(seg, return_counts=True)
        segment_lengths.extend(counts.tolist())
    
    mean_seg_len = float(np.mean(segment_lengths)) if segment_lengths else 1.0
    k = max(1, int(round(mean_seg_len / 2)))
    
    # Calculo metricas de segmentacion por documento
    # Estas metricas evaluan la calidad de la segmentacion como un todo
    pk_scores = []
    wd_scores = []
    for (_, _), group in df.groupby(["level", "doc_id"]):
        idx = group.index.to_numpy()
        y_true_doc = group["y"].to_numpy()
        y_pred_doc = y_pred[idx]
        
        # Convierto boundaries a segmentos para poder usar Pk
        seg_true = boundaries_to_segments(y_true_doc)
        seg_pred = boundaries_to_segments(y_pred_doc)
        
        pk_scores.append(pk(seg_true, seg_pred, k=k))
        wd_scores.append(windowdiff(y_true_doc, y_pred_doc, k=k))
    
    # Devuelvo todas las metricas en un diccionario
    return {
        "accuracy": accuracy,
        "precision_macro": precision_macro,
        "recall_macro": recall_macro,
        "f1_macro": f1_macro,
        "pk": float(np.mean(pk_scores)) if pk_scores else np.nan,
        "windowdiff": float(np.mean(wd_scores)) if wd_scores else np.nan,
        "k_window": k
    }


print("Funciones de metricas cargadas.")

In [ ]:
# Funciones auxiliares para gestionar modelos y resultados

def check_model_available(model: str) -> bool:
    """Verifico si un modelo esta disponible en Ollama."""
    try:
        # Hago un POST al endpoint show de Ollama para ver si el modelo existe
        resp = requests.post("http://localhost:11434/api/show", 
                            json={"name": model}, timeout=5)
        return resp.status_code == 200
    except:
        return False


def save_results(results: dict, path: Path):
    """Guardo resultados parciales (solo metricas, no predicciones completas).
    
    Esto me permite guardar progreso incremental por si el experimento falla
    a mitad de camino.
    """
    to_save = {
        name: {
            'metrics': r['metrics'],
            'classification_report': r['classification_report']
        }
        for name, r in results.items()
    }
    with open(path, 'w') as f:
        json.dump(to_save, f, indent=2)
    print(f"Resultados guardados en {path.name}")


print("Funciones auxiliares cargadas.")

## 1. Carga de datos y creacion del subset de evaluacion

In [ ]:
# Cargo los datos de oraciones procesadas desde los archivos JSONL
# Necesito los textos de las oraciones para construir los prompts

def load_sentences_dataframes() -> dict:
    """Cargo todos los DataFrames de oraciones procesadas.
    
    Returns:
        dict con clave (level, split) y valor DataFrame
        Cada DataFrame contiene las oraciones normalizadas con doc_id, sent_id, text_norm
    """
    sentences_dfs = {}
    # Cargo los 3 niveles (easy, medium, hard) x 2 splits (train, validation)
    for level in ['easy', 'medium', 'hard']:
        for split in ['train', 'validation']:
            path = DATA_PROCESSED / level / split / "sentences.jsonl"
            if path.exists():
                df = pd.read_json(path, lines=True)
                # Ordeno por doc_id y sent_id para mantener el orden original
                df = df.sort_values(['doc_id', 'sent_id']).reset_index(drop=True)
                sentences_dfs[(level, split)] = df
    return sentences_dfs


def get_sentence_text(sentences_dfs: dict, level: str, split: str, 
                      doc_id: str, sent_id: int) -> str:
    """Obtengo el texto de una oracion especifica desde el DataFrame procesado.
    
    Args:
        sentences_dfs: Diccionario de DataFrames cargados
        level: 'easy', 'medium', o 'hard'
        split: 'train' o 'validation'
        doc_id: ID del documento
        sent_id: ID de la oracion dentro del documento
    
    Returns:
        str: Texto normalizado de la oracion
    """
    key = (level, split)
    if key not in sentences_dfs:
        raise ValueError(f"No encuentro el DataFrame para {level}/{split}")
    
    df = sentences_dfs[key]
    # Busco la fila que coincide con doc_id y sent_id
    mask = (df['doc_id'] == doc_id) & (df['sent_id'] == sent_id)
    matches = df[mask]
    
    if len(matches) == 0:
        raise ValueError(f"No encuentro la oracion: {doc_id}, sent_id={sent_id}")
    
    return matches.iloc[0]['text_norm']


# Cargo las oraciones procesadas
print("Cargando oraciones procesadas...")
sentences_dfs = load_sentences_dataframes()
print(f"DataFrames cargados: {len(sentences_dfs)}")

# Cargo los datasets de boundaries (pares de oraciones consecutivas con etiquetas)
# Cada fila contiene:
# - level, doc_id, boundary_id: identificadores
# - sent_left_id, sent_right_id: IDs de las dos oraciones consecutivas
# - y: etiqueta (0 = mismo autor, 1 = cambio de autor)
boundaries_train = pd.read_csv(BOUNDARIES_DIR / "boundaries_train.csv")
boundaries_val = pd.read_csv(BOUNDARIES_DIR / "boundaries_validation.csv")

print(f"\nBoundaries train: {len(boundaries_train):,}")
print(f"Boundaries validation: {len(boundaries_val):,}")
print(f"\nDistribucion de clases en validation:")
print(boundaries_val['y'].value_counts())

In [ ]:
# Creo un subset estratificado para la evaluacion
# Uso estratificacion para mantener las proporciones de easy/medium/hard y clase 0/1

# Creo una clave de estratificacion combinando nivel y clase
# Esto asegura que el subset mantenga las proporciones de:
# 1. easy/medium/hard
# 2. clase 0 (mismo autor) / clase 1 (cambio)
boundaries_val['stratify_key'] = boundaries_val['level'] + '_' + boundaries_val['y'].astype(str)

# Muestreo estratificado: selecciono N_EVAL ejemplos manteniendo proporciones
eval_subset, _ = train_test_split(
    boundaries_val,
    train_size=N_EVAL,  # Solo tomo N_EVAL ejemplos del total
    stratify=boundaries_val['stratify_key'],  # Mantengo proporciones
    random_state=SEED  # Reproducibilidad
)

# Ordeno por nivel, documento y boundary_id para mantener consistencia
eval_subset = eval_subset.sort_values(['level', 'doc_id', 'boundary_id']).reset_index(drop=True)

print(f"\nSubset de evaluacion: {len(eval_subset)} ejemplos")
print(f"\nDistribucion por nivel:")
print(eval_subset['level'].value_counts())
print(f"\nDistribucion por clase:")
print(eval_subset['y'].value_counts())
print(f"\nTabla cruzada (nivel x clase):")
print(pd.crosstab(eval_subset['level'], eval_subset['y']))

## 2. Seleccion de ejemplos few-shot y construccion del prompt

In [ ]:
# Selecciono ejemplos few-shot del conjunto de entrenamiento
# Estos ejemplos se incluiran en el prompt para que el modelo aprenda en contexto

def select_few_shot_examples(boundaries_df: pd.DataFrame, n_shots: int, seed: int = 42) -> pd.DataFrame:
    """Selecciono n_shots ejemplos de cada clase, balanceados por nivel.
    
    Estrategia:
    - n_shots ejemplos de clase 0 (mismo autor)
    - n_shots ejemplos de clase 1 (cambio de autor)
    - Distribuidos equitativamente entre easy/medium/hard si es posible
    """
    np.random.seed(seed)
    examples = []
    
    # Para cada clase (0 = mismo autor, 1 = cambio)
    for y_class in [0, 1]:
        class_examples = boundaries_df[boundaries_df['y'] == y_class]
        
        # Intento balancear por nivel (easy/medium/hard)
        shots_per_level = n_shots // 3  # Divido entre los 3 niveles
        remaining = n_shots - (shots_per_level * 3)  # Shots restantes si no divide exacto
        
        for level in ['easy', 'medium', 'hard']:
            level_examples = class_examples[class_examples['level'] == level]
            if len(level_examples) > 0:
                # Anado 1 extra al primer nivel si n_shots no es divisible por 3
                n_sample = shots_per_level + (1 if remaining > 0 and level == 'easy' else 0)
                remaining -= 1 if remaining > 0 and level == 'easy' else 0
                # Sampleo aleatoriamente n_sample ejemplos de este nivel
                sampled = level_examples.sample(n=min(n_sample, len(level_examples)), random_state=seed)
                examples.append(sampled)
    
    return pd.concat(examples, ignore_index=True)


# Selecciono los ejemplos few-shot
few_shot_examples = select_few_shot_examples(boundaries_train, N_SHOTS, seed=SEED)

print(f"Ejemplos few-shot seleccionados: {len(few_shot_examples)}")
print(f"Distribucion por clase: {few_shot_examples['y'].value_counts().to_dict()}")
print(f"Distribucion por nivel: {few_shot_examples['level'].value_counts().to_dict()}")

# Cargo los textos reales de las oraciones para los ejemplos few-shot
# Necesito el texto para construir el prompt
few_shot_data = []
for _, row in few_shot_examples.iterrows():
    # Obtengo texto de oracion izquierda
    sent_left = get_sentence_text(sentences_dfs, row['level'], row['split'], 
                                   row['doc_id'], row['sent_left_id'])
    # Obtengo texto de oracion derecha
    sent_right = get_sentence_text(sentences_dfs, row['level'], row['split'], 
                                    row['doc_id'], row['sent_right_id'])
    # Convierto la etiqueta a formato yes/no para el prompt
    label = "yes" if row['y'] == 1 else "no"
    few_shot_data.append((sent_left, sent_right, label))

print(f"\nDatos few-shot cargados: {len(few_shot_data)} ejemplos")

In [ ]:
# Defino el prompt del sistema y las funciones para construir los prompts
# El prompt del sistema da las instrucciones generales al modelo

SYSTEM_PROMPT = """You are an expert forensic linguist specializing in authorship attribution and stylometry.

Your task: Determine if two consecutive sentences were written by the SAME author or DIFFERENT authors.

## Key stylometric features to analyze:
1. **Lexical features**: Vocabulary richness, word length distribution, use of rare words
2. **Syntactic features**: Sentence length, clause structure, use of subordination
3. **Punctuation patterns**: Comma usage, semicolons, dashes, exclamation marks
4. **Discourse markers**: Transitional phrases, hedging language, stance markers
5. **Formality level**: Contractions, colloquialisms, technical jargon
6. **Grammatical patterns**: Active vs passive voice, tense consistency, pronoun usage

## Important guidelines:
- IGNORE topic similarity - same author can write about different topics
- IGNORE factual content - focus ONLY on writing style
- Two sentences about the same topic can be from different authors
- Two sentences about different topics can be from the same author

## Response format:
Answer ONLY with 'yes' (author changed) or 'no' (same author).
Do not explain your reasoning unless asked."""


def build_few_shot_prompt(examples: list[tuple[str, str, str]]) -> str:
    """Construyo el prompt con ejemplos few-shot.
    
    Formato:
    Example 1:
    Sentence A: "..."
    Sentence B: "..."
    Author changed: yes/no
    """
    prompt_parts = []
    for i, (sent_a, sent_b, label) in enumerate(examples, 1):
        example_text = (
            f"Example {i}:\n"
            f'Sentence A: "{sent_a}"\n'
            f'Sentence B: "{sent_b}"\n'
            f"Author changed: {label}\n"
        )
        prompt_parts.append(example_text)
    return "\n".join(prompt_parts)


def get_extended_context(sentences_dfs: dict, level: str, split: str, 
                        doc_id: str, sent_id: int, offset: int) -> str:
    """Obtengo una oracion del contexto extendido.
    
    Args:
        offset: +1 para siguiente, -1 para anterior
    
    Esto me da mas contexto estilometrico al modelo en modo thinking.
    """
    try:
        return get_sentence_text(sentences_dfs, level, split, doc_id, sent_id + offset)
    except:
        # Si no existe (estamos al inicio/final del doc), devuelvo None
        return None


def build_query(sent_a: str, sent_b: str, few_shot_prompt: str, 
                use_cot: bool = False, 
                context_before: str = None, 
                context_after: str = None) -> str:
    """Construyo el prompt completo para una query.
    
    Args:
        sent_a, sent_b: Las dos oraciones a comparar
        few_shot_prompt: Ejemplos few-shot pre-formateados
        use_cot: Si True, pido razonamiento paso a paso (Chain-of-Thought)
        context_before: Oracion anterior a sent_a (contexto extendido)
        context_after: Oracion posterior a sent_b (contexto extendido)
    """
    # Construyo el texto de contexto extendido si esta disponible
    context_text = ""
    if context_before or context_after:
        context_text = "\n[Additional context for style analysis]\n"
        if context_before:
            context_text += f'Previous sentence: "{context_before}"\n'
        if context_after:
            context_text += f'Following sentence: "{context_after}"\n'
        context_text += "\n"
    
    if use_cot:
        # Chain-of-Thought: pido razonamiento explicito paso a paso
        # Esto ayuda al modelo a analizar mejor el estilo
        return (
            f"{few_shot_prompt}\n"
            f"{context_text}"
            f"Now analyze this pair:\n"
            f'Sentence A: "{sent_a}"\n'
            f'Sentence B: "{sent_b}"\n\n'
            f"Think step by step:\n"
            f"1. What is the formality level of each sentence?\n"
            f"2. How does the vocabulary compare?\n"
            f"3. Are sentence structures similar?\n"
            f"4. Based on style, did the author change?\n\n"
            f"Author changed (yes/no):"
        )
    else:
        # Prompt simple y directo (modo standard)
        return (
            f"{few_shot_prompt}\n"
            f"{context_text}"
            f"Now analyze this pair:\n"
            f'Sentence A: "{sent_a}"\n'
            f'Sentence B: "{sent_b}"\n'
            f"Author changed (yes/no):"
        )


# Construyo el prompt base con los ejemplos few-shot
few_shot_prompt = build_few_shot_prompt(few_shot_data)

print(f"Prompt construido:")
print(f"  SYSTEM_PROMPT: {len(SYSTEM_PROMPT)} caracteres")
print(f"  Ejemplos few-shot: {len(few_shot_data)}")

## 3. Funciones de inferencia con Ollama

In [ ]:
# Funciones para comunicarme con Ollama y procesar las respuestas

def query_ollama(model: str, prompt: str, system: str = SYSTEM_PROMPT, 
                 timeout: int = REQUEST_TIMEOUT, retries: int = MAX_RETRIES) -> str:
    """Consulto a Ollama con reintentos en caso de error.
    
    Args:
        model: Nombre del modelo (ej: qwen3:0.6b)
        prompt: El prompt a enviar
        system: Prompt del sistema (instrucciones generales)
        timeout: Timeout en segundos
        retries: Numero de reintentos si falla
    """
    # Preparo el payload para la API de Ollama
    payload = {
        "model": model,
        "prompt": prompt,
        "system": system,
        "stream": False,  # No quiero streaming, solo la respuesta completa
        "options": {
            "temperature": 0.0,  # Temperatura 0 para respuestas deterministas
            "top_p": 0.9,
            "repeat_penalty": 1.1  # Penalizo repeticiones
        }
    }
    
    # Intento hasta MAX_RETRIES veces si falla
    for attempt in range(retries):
        try:
            response = requests.post(OLLAMA_URL, json=payload, timeout=timeout)
            response.raise_for_status()  # Lanzo excepcion si status != 200
            return response.json()["response"].strip()
        except (requests.exceptions.RequestException, KeyError) as e:
            # Si es el ultimo intento, imprimo error y devuelvo "error"
            if attempt == retries - 1:
                print(f"Error al consultar {model} despues de {retries} intentos: {e}")
                return "error"
            continue
    return "error"


def parse_response(response: str) -> int:
    """Parseo la respuesta del modelo a 0/1. Devuelvo -1 si no puedo parsear.
    
    Soporta multiples formatos de respuesta:
    - "yes" / "no"
    - "Yes" / "No" 
    - "yes." / "no."
    - "1" / "0"
    - Respuestas con razonamiento que terminan en yes/no
    """
    response = response.lower().strip()
    
    # Busco yes/no al inicio o al final de la respuesta
    if response.startswith("yes") or response.endswith("yes") or " yes" in response:
        return 1  # Cambio de autor
    elif response.startswith("no") or response.endswith("no") or " no" in response:
        return 0  # Mismo autor
    elif response == "1":
        return 1
    elif response == "0":
        return 0
    
    # Busco patrones mas complejos
    if "different author" in response or "author changed" in response:
        return 1
    if "same author" in response or "author did not change" in response:
        return 0
    
    # Si no puedo parsear, devuelvo -1 (respuesta invalida)
    return -1


print("Funciones de inferencia cargadas.")

# Testeo la conexion con Ollama
try:
    test_response = requests.get("http://localhost:11434/api/version", timeout=5)
    if test_response.status_code == 200:
        print("Conexion con Ollama: OK")
    else:
        print(f"Advertencia: Ollama respondio con codigo {test_response.status_code}")
except Exception as e:
    print(f"Advertencia: No pude conectar con Ollama: {e}")
    print("Asegurate de que Ollama este ejecutandose antes de continuar.")

In [ ]:
# Funcion principal de evaluacion
# Esta funcion ejecuta el modelo sobre todo el subset y calcula metricas

def run_evaluation(model: str, subset_df: pd.DataFrame, few_shot_prompt: str, 
                   sentences_dfs: dict, use_cot: bool = False, 
                   use_extended_context: bool = False) -> dict:
    """Ejecuto la evaluacion de un modelo sobre el subset.
    
    Args:
        model: Nombre del modelo Ollama
        subset_df: DataFrame con los ejemplos a evaluar
        few_shot_prompt: Prompt con ejemplos few-shot pre-formateado
        sentences_dfs: Diccionario con DataFrames de oraciones
        use_cot: Chain-of-Thought (razonamiento explicito)
        use_extended_context: Incluir oraciones adicionales de contexto
    """
    predictions = []  # Aqui guardo las predicciones (0/1)
    responses_raw = []  # Aqui guardo las respuestas raw del modelo
    invalid_count = 0  # Contador de respuestas invalidas
    
    # Determino el modo para mostrarlo en los mensajes
    mode_desc = "Thinking" if (use_cot and use_extended_context) else "Standard"
    print(f"\nEvaluando {model} ({mode_desc}) sobre {len(subset_df)} ejemplos...")
    if use_cot:
        print("  Modo: Chain-of-Thought (razonamiento explicito)")
    if use_extended_context:
        print("  Modo: Contexto extendido (+-1 oracion)")
    
    # Itero sobre cada ejemplo del subset con una barra de progreso
    for idx, row in tqdm(subset_df.iterrows(), total=len(subset_df), desc=f"{model} ({mode_desc})"):
        # Cargo las dos oraciones desde el dataset procesado
        sent_left = get_sentence_text(sentences_dfs, row['level'], row['split'], 
                                       row['doc_id'], row['sent_left_id'])
        sent_right = get_sentence_text(sentences_dfs, row['level'], row['split'], 
                                        row['doc_id'], row['sent_right_id'])
        
        # Si esta activado el contexto extendido, cargo las oraciones adyacentes
        context_before = None
        context_after = None
        if use_extended_context:
            # Oracion anterior a sent_left
            context_before = get_extended_context(
                sentences_dfs, row['level'], row['split'], 
                row['doc_id'], row['sent_left_id'], -1
            )
            # Oracion posterior a sent_right
            context_after = get_extended_context(
                sentences_dfs, row['level'], row['split'], 
                row['doc_id'], row['sent_right_id'], +1
            )
        
        # Construyo el prompt completo y consulto al modelo
        query = build_query(
            sent_left, sent_right, few_shot_prompt, 
            use_cot=use_cot,
            context_before=context_before,
            context_after=context_after
        )
        response = query_ollama(model, query, system=SYSTEM_PROMPT)
        responses_raw.append(response)
        
        # Parseo la respuesta a 0/1
        pred = parse_response(response)
        if pred == -1:
            # Si no pude parsear, cuento como invalida y uso clase mayoritaria (0)
            invalid_count += 1
            pred = 0  # Fallback: predigo clase mayoritaria
        
        predictions.append(pred)
    
    predictions = np.array(predictions)
    
    # Calculo todas las metricas de evaluacion
    metrics = evaluate_predictions(subset_df, predictions)
    metrics['invalid_responses'] = invalid_count
    metrics['invalid_rate'] = invalid_count / len(subset_df)
    
    # Genero el classification report de sklearn
    y_true = subset_df['y'].to_numpy()
    clf_report = classification_report(y_true, predictions, zero_division=0)
    
    # Muestro un resumen de los resultados
    print(f"\n{model} ({mode_desc}) - Resultados:")
    print(f"  F1 macro:     {metrics['f1_macro']:.3f}")
    print(f"  Accuracy:     {metrics['accuracy']:.3f}")
    print(f"  Precision:    {metrics['precision_macro']:.3f}")
    print(f"  Recall:       {metrics['recall_macro']:.3f}")
    print(f"  Pk:           {metrics['pk']:.3f}")
    print(f"  WindowDiff:   {metrics['windowdiff']:.3f}")
    print(f"  Invalidas:    {invalid_count}/{len(subset_df)} ({metrics['invalid_rate']:.1%})")
    
    # Devuelvo todo en un diccionario
    return {
        'metrics': metrics,
        'predictions': predictions,
        'responses_raw': responses_raw,
        'classification_report': clf_report
    }


print("Funcion de evaluacion cargada.")

## 4. Evaluacion de modelos: qwen3:0.6b y qwen3:4b en modo Standard y Thinking

In [ ]:
# Inicializo el diccionario donde guardare los resultados de todos los modelos y modos
results = {}
print("Diccionario de resultados inicializado.")
print("Listo para evaluar los 4 modelos (0.6b y 4b, standard y thinking).")

### 4.1. Evaluacion de qwen3:0.6b

In [ ]:
# Evaluo qwen3:0.6b en modo STANDARD (sin thinking)
# Modo standard = prompt simple, sin Chain-of-Thought ni contexto extendido

model_name = "qwen3:0.6b"

if not check_model_available(model_name):
    print(f"{model_name} no disponible. Ejecuta: ollama pull {model_name}")
else:
    # Ejecuto la evaluacion en modo standard
    results[f"{model_name}_standard"] = run_evaluation(
        model_name, 
        eval_subset,  # El subset estratificado que cree antes
        few_shot_prompt,  # El prompt con los ejemplos few-shot
        sentences_dfs,  # Los DataFrames con los textos
        use_cot=False,  # NO uso Chain-of-Thought
        use_extended_context=False  # NO uso contexto extendido
    )
    
    # Guardo resultados intermedios (por si falla despues)
    timestamp = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
    save_results(results, REPORTS_DIR / f"11_icl_metrics_partial_{timestamp}.json")
    print(f"\n{model_name} (standard) completado.")

In [ ]:
# Evaluo qwen3:0.6b en modo THINKING (con CoT + contexto extendido)
# Modo thinking = prompt con razonamiento paso a paso + oraciones adyacentes

model_name = "qwen3:0.6b"

if not check_model_available(model_name):
    print(f"{model_name} no disponible. Ejecuta: ollama pull {model_name}")
else:
    # Ejecuto la evaluacion en modo thinking
    results[f"{model_name}_thinking"] = run_evaluation(
        model_name, 
        eval_subset, 
        few_shot_prompt, 
        sentences_dfs,
        use_cot=True,  # SI uso Chain-of-Thought (razonamiento paso a paso)
        use_extended_context=True  # SI uso contexto extendido (oraciones adyacentes)
    )
    
    # Guardo resultados intermedios
    timestamp = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
    save_results(results, REPORTS_DIR / f"11_icl_metrics_partial_{timestamp}.json")
    print(f"\n{model_name} (thinking) completado.")

### 4.2. Evaluacion de qwen3:4b

In [ ]:
# Evaluo qwen3:4b en modo STANDARD (sin thinking)
# Mismo procedimiento que con 0.6b, pero con el modelo mas grande

model_name = "qwen3:4b"

if not check_model_available(model_name):
    print(f"{model_name} no disponible. Ejecuta: ollama pull {model_name}")
else:
    # Ejecuto la evaluacion en modo standard
    results[f"{model_name}_standard"] = run_evaluation(
        model_name, 
        eval_subset,
        few_shot_prompt,
        sentences_dfs,
        use_cot=False,  # NO uso Chain-of-Thought
        use_extended_context=False  # NO uso contexto extendido
    )
    
    # Guardo resultados intermedios
    timestamp = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
    save_results(results, REPORTS_DIR / f"11_icl_metrics_partial_{timestamp}.json")
    print(f"\n{model_name} (standard) completado.")

In [ ]:
# Evaluo qwen3:4b en modo THINKING (con CoT + contexto extendido)
# Ultima evaluacion: modelo grande con thinking mode

model_name = "qwen3:4b"

if not check_model_available(model_name):
    print(f"{model_name} no disponible. Ejecuta: ollama pull {model_name}")
else:
    # Ejecuto la evaluacion en modo thinking
    results[f"{model_name}_thinking"] = run_evaluation(
        model_name, 
        eval_subset, 
        few_shot_prompt, 
        sentences_dfs,
        use_cot=True,  # SI uso Chain-of-Thought
        use_extended_context=True  # SI uso contexto extendido
    )
    
    # Guardo resultados intermedios
    timestamp = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
    save_results(results, REPORTS_DIR / f"11_icl_metrics_partial_{timestamp}.json")
    print(f"\n{model_name} (thinking) completado.")

### 4.3. Calculo de metricas por nivel de dificultad

Ahora calculo metricas desglosadas por nivel (easy, medium, hard) para analizar como varia el rendimiento segun la complejidad.

In [ ]:
# Funcion para calcular metricas separadas por nivel de dificultad

def evaluate_by_level(subset_df: pd.DataFrame, predictions: np.ndarray) -> dict:
    """Calculo metricas separadas por nivel de dificultad.
    
    Args:
        subset_df: DataFrame con columna 'level'
        predictions: Array con predicciones (0/1)
    
    Returns:
        dict: {'easy': {...}, 'medium': {...}, 'hard': {...}}
    """
    results_by_level = {}
    
    for level in ['easy', 'medium', 'hard']:
        # Filtro por nivel
        level_mask = subset_df['level'] == level
        level_df = subset_df[level_mask].reset_index(drop=True)
        level_preds = predictions[level_mask]
        
        if len(level_df) == 0:
            continue
        
        # Calculo metricas para este nivel usando la funcion existente
        metrics = evaluate_predictions(level_df, level_preds)
        
        # Anado info extra
        metrics['n_examples'] = len(level_df)
        metrics['n_class_0'] = (level_df['y'] == 0).sum()
        metrics['n_class_1'] = (level_df['y'] == 1).sum()
        
        results_by_level[level] = metrics
    
    return results_by_level

print("Funcion evaluate_by_level() definida.")

In [ ]:
# Calculo metricas por nivel para cada modelo evaluado

print("Calculando metricas por nivel para todos los modelos...")

results_by_level = {}
for model_name, result in results.items():
    results_by_level[model_name] = evaluate_by_level(eval_subset, result['predictions'])
    
print(f"✓ Metricas por nivel calculadas para {len(results_by_level)} modelos.")

## 5. Resumen de resultados y comparacion

In [ ]:
# Creo tablas resumen: agregada + por nivel

if not results:
    print("No hay resultados todavia. Ejecuta las celdas de evaluacion primero.")
else:
    # ============================================
    # TABLA 1: RESUMEN AGREGADO
    # ============================================
    summary_data = []
    for model_name, result in results.items():
        m = result['metrics']
        parts = model_name.rsplit('_', 1)
        model = parts[0]
        mode = parts[1].capitalize()
        
        summary_data.append({
            'Modelo': model,
            'Modo': mode,
            'F1 Macro': m['f1_macro'],
            'Accuracy': m['accuracy'],
            'Precision': m['precision_macro'],
            'Recall': m['recall_macro'],
            'Pk': m['pk'],
            'WindowDiff': m['windowdiff'],
            'Invalidas %': f"{m['invalid_rate']:.1%}"
        })
    
    summary_df = pd.DataFrame(summary_data)
    summary_df = summary_df.sort_values(['Modelo', 'Modo'])
    
    print("="*100)
    print("RESUMEN AGREGADO: Comparacion de modelos (0.6b y 4b) x modos (Standard y Thinking)")
    print("="*100)
    print(summary_df.to_string(index=False))
    print("="*100)
    
    # Impacto del thinking mode
    print("\nImpacto del modo Thinking por modelo (Delta F1 Macro):")
    for model in MODELS:
        standard_f1 = summary_df[(summary_df['Modelo'] == model) & (summary_df['Modo'] == 'Standard')]['F1 Macro']
        thinking_f1 = summary_df[(summary_df['Modelo'] == model) & (summary_df['Modo'] == 'Thinking')]['F1 Macro']
        
        if len(standard_f1) > 0 and len(thinking_f1) > 0:
            standard_f1 = standard_f1.iloc[0]
            thinking_f1 = thinking_f1.iloc[0]
            delta = thinking_f1 - standard_f1
            delta_pct = (delta / standard_f1 * 100) if standard_f1 > 0 else 0
            print(f"  {model}: {delta:+.3f} ({delta_pct:+.1f}%)")
    
    # ============================================
    # TABLA 2: RESUMEN POR NIVEL
    # ============================================
    summary_level_data = []
    for model_name, levels_data in results_by_level.items():
        parts = model_name.rsplit('_', 1)
        model = parts[0]
        mode = parts[1].capitalize()
        
        for level, metrics in levels_data.items():
            summary_level_data.append({
                'Modelo': model,
                'Modo': mode,
                'Nivel': level.capitalize(),
                'N': metrics['n_examples'],
                'F1 Macro': metrics['f1_macro'],
                'Accuracy': metrics['accuracy'],
                'Pk': metrics['pk'],
                'WindowDiff': metrics['windowdiff'],
            })
    
    summary_level_df = pd.DataFrame(summary_level_data)
    summary_level_df = summary_level_df.sort_values(['Nivel', 'Modelo', 'Modo'])
    
    print("\n" + "="*100)
    print("RESUMEN POR NIVEL DE DIFICULTAD")
    print("="*100)
    print(summary_level_df.to_string(index=False))
    print("="*100)
    
    # ============================================
    # ANÁLISIS: Impacto de thinking por nivel
    # ============================================
    print("\n" + "="*100)
    print("IMPACTO DEL MODO THINKING POR NIVEL")
    print("="*100)
    
    for model in MODELS:
        print(f"\n{model}:")
        std_key = f"{model}_standard"
        think_key = f"{model}_thinking"
        
        if std_key in results_by_level and think_key in results_by_level:
            for level in ['easy', 'medium', 'hard']:
                if level in results_by_level[std_key] and level in results_by_level[think_key]:
                    f1_std = results_by_level[std_key][level]['f1_macro']
                    f1_think = results_by_level[think_key][level]['f1_macro']
                    delta = f1_think - f1_std
                    delta_pct = (delta / f1_std * 100) if f1_std > 0 else 0
                    
                    print(f"  {level.capitalize():6s}: Standard={f1_std:.3f}, Thinking={f1_think:.3f}, "
                          f"Δ={delta:+.3f} ({delta_pct:+.1f}%)")
    print("="*100)

In [ ]:
# Comparo los resultados ICL con los modelos entrenados del experimento E3

e3_metrics_path = REPORTS_DIR / "09_metrics.json"

if e3_metrics_path.exists():
    with open(e3_metrics_path, "r") as f:
        e3_metrics = json.load(f)
    
    # Tabla de comparación agregada
    comparison_data = []
    
    for name in ['bert_cnn', 'bert_lstm', 'w2v_mlp', 'tfidf_svm']:
        if name in e3_metrics and 'metrics_val' in e3_metrics[name]:
            m = e3_metrics[name]['metrics_val']
            comparison_data.append({
                'Modelo': f"E3: {name}",
                'Tipo': 'Entrenado',
                'F1 Macro': m['f1_macro'],
                'Accuracy': m['accuracy'],
                'Pk': m['pk'],
                'WindowDiff': m['windowdiff']
            })
    
    for model_name, result in results.items():
        m = result['metrics']
        parts = model_name.rsplit('_', 1)
        model = parts[0]
        mode = parts[1].capitalize()
        
        comparison_data.append({
            'Modelo': f"ICL: {model} ({mode})",
            'Tipo': 'Zero-shot',
            'F1 Macro': m['f1_macro'],
            'Accuracy': m['accuracy'],
            'Pk': m['pk'],
            'WindowDiff': m['windowdiff']
        })
    
    comparison_df = pd.DataFrame(comparison_data)
    comparison_df = comparison_df.sort_values('F1 Macro', ascending=False)
    
    print("\n" + "="*100)
    print("COMPARACION: E3 (entrenados) vs ICL (zero-shot)")
    print("="*100)
    print(comparison_df.to_string(index=False))
    print("="*100)
    
    print("\nNota:")
    print("  - E3 evaluado en validation completo (33,858 ejemplos)")
    print(f"  - ICL evaluado en subset estratificado ({N_EVAL} ejemplos)")
else:
    print(f"No encuentro las metricas de E3 en {e3_metrics_path}")

## 6. Visualizaciones

In [ ]:
# Grafico de F1 Macro por nivel (3 subplots)

if len(results) >= 2:
    fig, axes = plt.subplots(1, 3, figsize=(16, 5))
    
    levels = ['easy', 'medium', 'hard']
    model_labels = ['0.6b', '4b']
    x = np.arange(len(model_labels))
    width = 0.35
    
    for idx, level in enumerate(levels):
        ax = axes[idx]
        
        # Extraigo F1 para este nivel
        f1_standard = []
        f1_thinking = []
        
        for model in MODELS:
            std_key = f"{model}_standard"
            think_key = f"{model}_thinking"
            
            f1_std = results_by_level.get(std_key, {}).get(level, {}).get('f1_macro', 0)
            f1_think = results_by_level.get(think_key, {}).get(level, {}).get('f1_macro', 0)
            
            f1_standard.append(f1_std)
            f1_thinking.append(f1_think)
        
        # Creo barras
        bars1 = ax.bar(x - width/2, f1_standard, width, label='Standard', color='#4ECDC4', alpha=0.8)
        bars2 = ax.bar(x + width/2, f1_thinking, width, label='Thinking', color='#FF6B6B', alpha=0.8)
        
        ax.set_ylabel('F1 Macro', fontsize=11)
        ax.set_title(f'Nivel: {level.upper()}', fontsize=12, fontweight='bold')
        ax.set_xticks(x)
        ax.set_xticklabels(model_labels)
        ax.legend()
        ax.grid(axis='y', alpha=0.3)
        
        # Valores en barras
        for bar in bars1 + bars2:
            height = bar.get_height()
            if height > 0:
                ax.text(bar.get_x() + bar.get_width()/2., height + 0.01,
                       f'{height:.3f}', ha='center', va='bottom', fontsize=9)
    
    plt.tight_layout()
    plt.savefig(REPORTS_DIR / '11_f1_by_level.png', dpi=300, bbox_inches='tight')
    plt.show()
    print(f"Grafico guardado en {REPORTS_DIR / '11_f1_by_level.png'}")
else:
    print("Necesito al menos 2 resultados para crear el grafico.")

In [ ]:
# Graficos agregados: F1 y Pk

if len(results) >= 2:
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
    
    models_data = {'qwen3:0.6b': {}, 'qwen3:4b': {}}
    
    for model_name, result in results.items():
        parts = model_name.rsplit('_', 1)
        model = parts[0]
        mode = parts[1]
        
        if model in models_data:
            models_data[model][mode] = result['metrics']
    
    model_labels = ['0.6b', '4b']
    x = np.arange(len(model_labels))
    width = 0.35
    
    f1_standard = [models_data['qwen3:0.6b'].get('standard', {}).get('f1_macro', 0),
                   models_data['qwen3:4b'].get('standard', {}).get('f1_macro', 0)]
    f1_thinking = [models_data['qwen3:0.6b'].get('thinking', {}).get('f1_macro', 0),
                   models_data['qwen3:4b'].get('thinking', {}).get('f1_macro', 0)]
    
    pk_standard = [models_data['qwen3:0.6b'].get('standard', {}).get('pk', 0),
                   models_data['qwen3:4b'].get('standard', {}).get('pk', 0)]
    pk_thinking = [models_data['qwen3:0.6b'].get('thinking', {}).get('pk', 0),
                   models_data['qwen3:4b'].get('thinking', {}).get('pk', 0)]
    
    # Grafico 1: F1 Macro
    bars1_std = ax1.bar(x - width/2, f1_standard, width, label='Standard', color='#4ECDC4', alpha=0.8)
    bars1_think = ax1.bar(x + width/2, f1_thinking, width, label='Thinking', color='#FF6B6B', alpha=0.8)
    
    ax1.set_ylabel('F1 Macro Score', fontsize=11)
    ax1.set_title('F1 Macro AGREGADO', fontsize=12, fontweight='bold')
    ax1.set_xticks(x)
    ax1.set_xticklabels(model_labels)
    ax1.legend()
    ax1.grid(axis='y', alpha=0.3)
    
    for bar in bars1_std + bars1_think:
        height = bar.get_height()
        ax1.text(bar.get_x() + bar.get_width()/2., height + 0.005,
                f'{height:.3f}', ha='center', va='bottom', fontsize=9)
    
    # Grafico 2: Pk
    bars2_std = ax2.bar(x - width/2, pk_standard, width, label='Standard', color='#4ECDC4', alpha=0.8)
    bars2_think = ax2.bar(x + width/2, pk_thinking, width, label='Thinking', color='#FF6B6B', alpha=0.8)
    
    ax2.set_ylabel('Pk Score (menor es mejor)', fontsize=11)
    ax2.set_title('Pk AGREGADO', fontsize=12, fontweight='bold')
    ax2.set_xticks(x)
    ax2.set_xticklabels(model_labels)
    ax2.legend()
    ax2.grid(axis='y', alpha=0.3)
    
    for bar in bars2_std + bars2_think:
        height = bar.get_height()
        ax2.text(bar.get_x() + bar.get_width()/2., height + 0.005,
                f'{height:.3f}', ha='center', va='bottom', fontsize=9)
    
    plt.tight_layout()
    plt.savefig(REPORTS_DIR / '11_mode_comparison_aggregated.png', dpi=300, bbox_inches='tight')
    plt.show()
    print(f"Grafico guardado en {REPORTS_DIR / '11_mode_comparison_aggregated.png'}")
else:
    print("Necesito al menos 2 resultados para crear el grafico.")

In [ ]:
# Heatmap de metricas agregadas

if results:
    metrics_data = []
    index_labels = []
    
    for model_name in sorted(results.keys()):
        result = results[model_name]
        parts = model_name.rsplit('_', 1)
        model = parts[0].replace('qwen3:', '')
        mode = "Std" if parts[1] == 'standard' else "Think"
        index_labels.append(f"{model} {mode}")
        
        m = result['metrics']
        metrics_data.append([
            m['f1_macro'],
            m['accuracy'],
            m['precision_macro'],
            m['recall_macro'],
            1 - m['pk'],
            1 - m['windowdiff']
        ])
    
    metrics_df = pd.DataFrame(
        metrics_data,
        index=index_labels,
        columns=['F1', 'Accuracy', 'Precision', 'Recall', '1-Pk', '1-WD']
    )
    
    fig, ax = plt.subplots(figsize=(8, 6))
    im = ax.imshow(metrics_df.values, cmap='RdYlGn', aspect='auto', vmin=0, vmax=1)
    
    ax.set_xticks(np.arange(len(metrics_df.columns)))
    ax.set_yticks(np.arange(len(metrics_df.index)))
    ax.set_xticklabels(metrics_df.columns)
    ax.set_yticklabels(metrics_df.index)
    
    plt.setp(ax.get_xticklabels(), rotation=0, ha="center")
    
    for i in range(len(metrics_df.index)):
        for j in range(len(metrics_df.columns)):
            text = ax.text(j, i, f'{metrics_df.values[i, j]:.2f}',
                          ha="center", va="center", color="black", fontsize=9)
    
    ax.set_title('Heatmap AGREGADO', fontweight='bold', fontsize=12)
    fig.colorbar(im, ax=ax)
    plt.tight_layout()
    plt.savefig(REPORTS_DIR / '11_metrics_heatmap_aggregated.png', dpi=300, bbox_inches='tight')
    plt.show()
    print(f"Heatmap guardado en {REPORTS_DIR / '11_metrics_heatmap_aggregated.png'}")

In [ ]:
# Heatmap de F1 por nivel

if results_by_level:
    f1_by_level_data = []
    config_labels = []
    
    for model_name in sorted(results_by_level.keys()):
        parts = model_name.rsplit('_', 1)
        model = parts[0].replace('qwen3:', '')
        mode = "Std" if parts[1] == 'standard' else "Think"
        config_labels.append(f"{model} {mode}")
        
        levels_data = results_by_level[model_name]
        f1_by_level_data.append([
            levels_data.get('easy', {}).get('f1_macro', 0),
            levels_data.get('medium', {}).get('f1_macro', 0),
            levels_data.get('hard', {}).get('f1_macro', 0),
        ])
    
    f1_level_df = pd.DataFrame(
        f1_by_level_data,
        index=config_labels,
        columns=['Easy', 'Medium', 'Hard']
    )
    
    fig, ax = plt.subplots(figsize=(6, 6))
    im = ax.imshow(f1_level_df.values, cmap='RdYlGn', aspect='auto', vmin=0, vmax=max(0.5, f1_level_df.values.max()))
    
    ax.set_xticks(np.arange(len(f1_level_df.columns)))
    ax.set_yticks(np.arange(len(f1_level_df.index)))
    ax.set_xticklabels(f1_level_df.columns)
    ax.set_yticklabels(f1_level_df.index)
    
    plt.setp(ax.get_xticklabels(), rotation=0, ha="center")
    
    for i in range(len(f1_level_df.index)):
        for j in range(len(f1_level_df.columns)):
            text = ax.text(j, i, f'{f1_level_df.values[i, j]:.3f}',
                          ha="center", va="center", color="black", fontsize=10)
    
    ax.set_title('Heatmap F1 Macro POR NIVEL', fontweight='bold', fontsize=12)
    fig.colorbar(im, ax=ax)
    plt.tight_layout()
    plt.savefig(REPORTS_DIR / '11_f1_heatmap_by_level.png', dpi=300, bbox_inches='tight')
    plt.show()
    print(f"Heatmap guardado en {REPORTS_DIR / '11_f1_heatmap_by_level.png'}")

## 7. Guardado de resultados finales

In [ ]:
# Guardo las metricas finales: agregadas + por nivel

metrics_to_save = {}

for model_name, result in results.items():
    metrics_to_save[model_name] = {
        'metrics_aggregated': result['metrics'],
        'metrics_by_level': results_by_level[model_name],
        'classification_report': result['classification_report']
    }

# Metadatos
metrics_to_save['_metadata'] = {
    'models': MODELS,
    'n_eval': N_EVAL,
    'n_shots': N_SHOTS,
    'seed': SEED,
    'modes': ['standard', 'thinking'],
    'eval_subset_distribution': eval_subset['y'].value_counts().to_dict(),
    'eval_subset_by_level': {
        level: len(eval_subset[eval_subset['level'] == level])
        for level in ['easy', 'medium', 'hard']
    },
    'note': 'Evaluacion ICL comparando 2 modelos (qwen3:0.6b y qwen3:4b) en 2 modos cada uno (standard vs thinking con CoT + contexto extendido). Incluye metricas agregadas y desglosadas por nivel de dificultad.'
}

with open(REPORTS_DIR / "11_icl_metrics_final.json", "w") as f:
    json.dump(metrics_to_save, f, indent=2)

print(f"Metricas finales guardadas en {REPORTS_DIR / '11_icl_metrics_final.json'}")
print("  - Incluye metricas agregadas")
print("  - Incluye metricas por nivel (easy, medium, hard)")

In [ ]:
# Guardo las predicciones en formato .npz

for model_name, result in results.items():
    filename = f"icl_{model_name.replace(':', '_')}_val.npz"
    
    np.savez_compressed(
        PREDICTIONS_DIR / filename,
        predictions=result['predictions'],
        y_true=eval_subset['y'].to_numpy(),
        doc_ids=eval_subset['doc_id'].to_numpy(),
        boundary_ids=eval_subset['boundary_id'].to_numpy(),
        levels=eval_subset['level'].to_numpy(),  # Incluyo niveles para analisis posterior
        responses_raw=np.array(result['responses_raw'], dtype=object)
    )
    
    print(f"Predicciones guardadas en {filename}")

print(f"\nTodas las predicciones guardadas en {PREDICTIONS_DIR}")

## Conclusion

En este notebook he evaluado el impacto del modo thinking (Chain-of-Thought + contexto extendido) en el rendimiento de ICL para atribucion de autoria, comparando dos modelos de diferente tamano (qwen3:0.6b y qwen3:4b).

Los resultados clave estan visibles en las tablas de comparacion y graficos anteriores, tanto agregados como desglosados por nivel de dificultad (easy, medium, hard).